# Flattening the Power BI activity log

This notebook takes the raw `activity_events.json` produced by `smoke_test.py`
and turns it into a flat, analysis-ready table, in Python and in SQL.

**Contents**

1. Flatten the JSON in Python
2. Flatten the same JSON in SQL, for Snowflake, Databricks and Fabric
3. Keep only report-related activity
4. Group activities into buckets by type

**About the activity data.** Every event is a flat JSON object already, but the
set of keys varies from event to event. A `ViewReport` record carries report and
workspace fields; an admin record carries almost none. So "flattening" here mostly
means taking the union of every key seen across the batch and producing one wide
table with nulls where a field doesn't apply. A couple of fields (`ModelsSnapshots`,
`OrgAppPermission`) do hold nested arrays, and those get handled explicitly.

**Source for operation names.** The operation names and their friendly descriptions
come from Microsoft's [Fabric operation list](https://learn.microsoft.com/en-us/fabric/admin/operation-list),
saved alongside this notebook as `powerbi_operations_reference.csv`.

## Setup

In [ ]:
import json
import re
from collections import Counter
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

SRC = Path("activity_events.json")
REFERENCE = Path("powerbi_operations_reference.csv")

raw = json.loads(SRC.read_text(encoding="utf-8"))
print(f"{len(raw):,} raw events loaded from {SRC}")
print(f"{len({k for r in raw for k in r})} distinct field names across the batch")

---

## 1. Flatten the JSON in Python

Two ways to do this. The first is a hand-written flattener with no dependencies
beyond the standard library, which is useful when you want to control exactly how
nested values are collapsed. The second is a one-liner with pandas.

### 1a. Standard-library flattener

This walks each record and produces `parent_child` style column names. Its main
job is deciding what to do with nested values:

- A nested **object** becomes several columns, one per inner key.
- A nested **array of simple values** becomes a single comma-joined string, so it
  still fits in one cell.
- A nested **array of objects** gets one set of columns per element, numbered.
- An **empty array** becomes null, rather than an empty string, so it doesn't get
  confused with a real value.

In [ ]:
def flatten_record(record, parent_key="", sep="_"):
    """Collapse one nested JSON object into a single flat dictionary."""
    out = {}
    for key, value in record.items():
        name = f"{parent_key}{sep}{key}" if parent_key else key

        if isinstance(value, dict):
            out.update(flatten_record(value, name, sep))

        elif isinstance(value, list):
            if not value:
                out[name] = None
            elif all(not isinstance(v, (dict, list)) for v in value):
                out[name] = ", ".join(str(v) for v in value)
            else:
                for i, item in enumerate(value):
                    if isinstance(item, dict):
                        out.update(flatten_record(item, f"{name}{sep}{i}", sep))
                    else:
                        out[f"{name}{sep}{i}"] = item
        else:
            out[name] = value

    return out


flat_rows = [flatten_record(record) for record in raw]
events = pd.DataFrame(flat_rows)

print(f"shape: {events.shape[0]} rows x {events.shape[1]} columns")
events.head(3)

### 1b. The pandas one-liner

`json_normalize` does the same job in a single call. It's shorter, and it's the
right choice when the shape is simple. The hand-written version above is worth
keeping when you need arrays collapsed a particular way, because `json_normalize`
leaves list values as Python lists sitting inside cells, which will not survive a
trip to CSV cleanly.

In [ ]:
normalized = pd.json_normalize(raw, sep="_")
print(f"json_normalize shape: {normalized.shape}")

# Columns where json_normalize left an actual list object in the cell:
list_cols = [c for c in normalized.columns
             if normalized[c].apply(lambda v: isinstance(v, list)).any()]
print("columns still holding list objects:", list_cols)

### 1c. Fix the data types

Everything arrives as text. Three things are worth correcting before analysis:

- `CreationTime` is an ISO timestamp string, and it is **always UTC** even though
  it carries no timezone marker. Parsing it as UTC now avoids a whole category of
  off-by-a-few-hours mistakes later.
- `IsSuccess` is a real boolean but is missing on some records.
- Adding plain date and hour columns makes grouping much easier.

In [ ]:
events["CreationTime"] = pd.to_datetime(events["CreationTime"], utc=True, errors="coerce")
events["ActivityDate"] = events["CreationTime"].dt.date
events["ActivityHour"] = events["CreationTime"].dt.hour

if "IsSuccess" in events.columns:
    events["IsSuccess"] = events["IsSuccess"].astype("boolean")

print(events[["CreationTime", "ActivityDate", "Operation", "UserId"]].head())
print()
print("time span:", events["CreationTime"].min(), "->", events["CreationTime"].max())

### 1d. Save the flat table

CSV for a quick look in Excel, Parquet for anything downstream. Parquet is worth
preferring where you can, because it keeps the data types you just fixed. CSV
throws them away and you get to re-parse the timestamps next time.

In [ ]:
events.to_csv("activity_events_flat.csv", index=False)
print("wrote activity_events_flat.csv")

try:
    events.to_parquet("activity_events_flat.parquet", index=False)
    print("wrote activity_events_flat.parquet")
except Exception as exc:
    print(f"parquet skipped ({type(exc).__name__}); install pyarrow to enable it")

---

## 2. Flatten the same JSON in SQL

The three platforms below all follow the same two-step shape:

1. **Land** the file with its structure intact, in a single column typed for
   semi-structured data.
2. **Project** the fields you want out of that column into a proper table or view.

Keeping those steps separate matters here. The activity log adds new fields over
time, and if you land the raw payload first, a new field is a change to step two
only. You never have to reload history.

The cells below are SQL for you to paste into each platform, not Python to run in
this notebook.

### 2a. Snowflake

Snowflake's `VARIANT` type holds the whole JSON object. `STRIP_OUTER_ARRAY` tells
the loader that the file is one big array and each element should become its own
row, which saves you a flattening step.

```sql
-- Step 1: land the raw payload -------------------------------------------
CREATE OR REPLACE FILE FORMAT ff_json
    TYPE = JSON
    STRIP_OUTER_ARRAY = TRUE;

CREATE OR REPLACE STAGE stg_pbi_activity
    FILE_FORMAT = ff_json;

-- From SnowSQL or the Snowsight file upload:
--   PUT file://C:/Users/asher/OneDrive/Desktop/code/activity_events.json @stg_pbi_activity;

CREATE OR REPLACE TABLE pbi_activity_raw (
    payload      VARIANT,
    loaded_at    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

COPY INTO pbi_activity_raw (payload)
FROM @stg_pbi_activity
FILE_FORMAT = (FORMAT_NAME = ff_json)
ON_ERROR = ABORT_STATEMENT;


-- Step 2: project the fields ----------------------------------------------
CREATE OR REPLACE VIEW pbi_activity AS
SELECT
    payload:Id::STRING                                AS event_id,
    TO_TIMESTAMP_NTZ(payload:CreationTime::STRING)    AS creation_time_utc,
    payload:Operation::STRING                         AS operation,
    payload:Activity::STRING                          AS activity,
    payload:RecordType::NUMBER                        AS record_type,
    payload:UserId::STRING                            AS user_id,
    payload:UserType::NUMBER                          AS user_type,
    payload:ClientIP::STRING                          AS client_ip,
    payload:UserAgent::STRING                         AS user_agent,
    payload:IsSuccess::BOOLEAN                        AS is_success,
    payload:WorkspaceId::STRING                       AS workspace_id,
    payload:WorkSpaceName::STRING                     AS workspace_name,
    payload:ReportId::STRING                          AS report_id,
    payload:ReportName::STRING                        AS report_name,
    payload:ReportType::STRING                        AS report_type,
    payload:DatasetId::STRING                         AS dataset_id,
    payload:DatasetName::STRING                       AS dataset_name,
    payload:ArtifactId::STRING                        AS artifact_id,
    payload:ArtifactName::STRING                      AS artifact_name,
    payload:ArtifactKind::STRING                      AS artifact_kind,
    payload:AppName::STRING                           AS app_name,
    payload:CapacityId::STRING                        AS capacity_id,
    payload:CapacityName::STRING                      AS capacity_name,
    payload:DistributionMethod::STRING                AS distribution_method,
    payload:ConsumptionMethod::STRING                 AS consumption_method,
    payload                                           AS full_payload
FROM pbi_activity_raw;
```

**If you loaded the file as a single blob instead** (no `STRIP_OUTER_ARRAY`), use
`LATERAL FLATTEN` to turn the array into rows:

```sql
SELECT
    element.value:Id::STRING        AS event_id,
    element.value:Operation::STRING AS operation
FROM pbi_activity_raw,
     LATERAL FLATTEN(input => payload) AS element;
```

`LATERAL FLATTEN` is also what you need for the genuinely nested fields, such as
turning each entry of `ModelsSnapshots` into its own row:

```sql
SELECT
    payload:Id::STRING     AS event_id,
    snapshot.value         AS model_snapshot
FROM pbi_activity_raw,
     LATERAL FLATTEN(input => payload:ModelsSnapshots, OUTER => TRUE) AS snapshot;
```

`OUTER => TRUE` keeps events whose array is empty. Without it those events vanish
from the result, which is an easy way to quietly lose rows.

### 2b. Databricks

Spark infers the schema and flattens the top level for you, so there's less to
write. The one option that matters is `multiLine`, because the file is a single
pretty-printed JSON array rather than one object per line. Leave it off and you'll
get a parse error or a single corrupt row.

```sql
-- Step 1: land the raw payload -------------------------------------------
CREATE OR REPLACE TEMP VIEW pbi_activity_raw
USING json
OPTIONS (
    path      '/Volumes/main/default/landing/activity_events.json',
    multiLine 'true'
);

-- Step 2: project the fields ----------------------------------------------
CREATE OR REPLACE TABLE main.default.pbi_activity AS
SELECT
    Id                                             AS event_id,
    CAST(CreationTime AS TIMESTAMP)                AS creation_time_utc,
    Operation                                      AS operation,
    Activity                                       AS activity,
    RecordType                                     AS record_type,
    UserId                                         AS user_id,
    ClientIP                                       AS client_ip,
    UserAgent                                      AS user_agent,
    IsSuccess                                      AS is_success,
    WorkspaceId                                    AS workspace_id,
    WorkSpaceName                                  AS workspace_name,
    ReportId                                       AS report_id,
    ReportName                                     AS report_name,
    ReportType                                     AS report_type,
    DatasetId                                      AS dataset_id,
    DatasetName                                    AS dataset_name,
    ArtifactId                                     AS artifact_id,
    ArtifactName                                   AS artifact_name,
    ArtifactKind                                   AS artifact_kind,
    AppName                                        AS app_name,
    CapacityId                                     AS capacity_id,
    CapacityName                                   AS capacity_name,
    DistributionMethod                             AS distribution_method,
    ConsumptionMethod                              AS consumption_method
FROM pbi_activity_raw;
```

**Keeping the payload intact instead.** If you'd rather land the raw text and
project later, read the file as one string column and parse on read. `variant` is
the modern option on recent runtimes, `from_json` works everywhere:

```sql
CREATE OR REPLACE TEMP VIEW pbi_activity_variant AS
SELECT parse_json(payload) AS payload
FROM text.`/Volumes/main/default/landing/activity_events.json`;

SELECT
    payload:Id::string        AS event_id,
    payload:Operation::string AS operation
FROM pbi_activity_variant;
```

**Exploding a nested array** works the same way as Snowflake's flatten, using
`explode_outer` so empty arrays don't drop their parent row:

```sql
SELECT
    Id AS event_id,
    snapshot
FROM pbi_activity_raw
LATERAL VIEW explode_outer(ModelsSnapshots) AS snapshot;
```

**PySpark equivalent**, if you prefer the DataFrame API:

```python
df = (spark.read
        .option("multiLine", "true")
        .json("/Volumes/main/default/landing/activity_events.json"))

df.selectExpr(
    "Id AS event_id",
    "CAST(CreationTime AS TIMESTAMP) AS creation_time_utc",
    "Operation AS operation",
    "UserId AS user_id",
).write.mode("overwrite").saveAsTable("main.default.pbi_activity")
```

### 2c. Microsoft Fabric

Fabric gives you two routes, and which one you want depends on where the file
lands.

**Route 1: Warehouse or SQL analytics endpoint, using `OPENJSON`.** This is
T-SQL, so it looks quite different from the other two. `OPENROWSET` reads the file
as one big string, and `OPENJSON` with a `WITH` clause pulls typed columns out of
it. The odd `0x0b` delimiters are a standard trick to stop the CSV reader from
splitting the file at all, so the whole document arrives in one cell.

```sql
-- Step 1: read the file as a single string --------------------------------
CREATE OR ALTER VIEW dbo.pbi_activity_raw AS
SELECT payload
FROM OPENROWSET(
        BULK 'https://onelake.dfs.fabric.microsoft.com/<workspace>/<lakehouse>.Lakehouse/Files/activity_events.json',
        FORMAT          = 'CSV',
        FIELDQUOTE      = '0x0b',
        FIELDTERMINATOR = '0x0b',
        ROWTERMINATOR   = '0x0b'
     ) WITH (payload VARCHAR(MAX)) AS src;


-- Step 2: shred the array into typed rows ---------------------------------
CREATE OR ALTER VIEW dbo.pbi_activity AS
SELECT evt.*
FROM dbo.pbi_activity_raw AS r
CROSS APPLY OPENJSON(r.payload)
WITH (
    event_id             VARCHAR(50)   '$.Id',
    creation_time_utc    DATETIME2(3)  '$.CreationTime',
    operation            VARCHAR(200)  '$.Operation',
    activity             VARCHAR(200)  '$.Activity',
    record_type          INT           '$.RecordType',
    user_id              VARCHAR(250)  '$.UserId',
    client_ip            VARCHAR(50)   '$.ClientIP',
    user_agent           VARCHAR(1000) '$.UserAgent',
    is_success           BIT           '$.IsSuccess',
    workspace_id         VARCHAR(50)   '$.WorkspaceId',
    workspace_name       VARCHAR(400)  '$.WorkSpaceName',
    report_id            VARCHAR(50)   '$.ReportId',
    report_name          VARCHAR(400)  '$.ReportName',
    report_type          VARCHAR(100)  '$.ReportType',
    dataset_id           VARCHAR(50)   '$.DatasetId',
    dataset_name         VARCHAR(400)  '$.DatasetName',
    artifact_id          VARCHAR(50)   '$.ArtifactId',
    artifact_name        VARCHAR(400)  '$.ArtifactName',
    artifact_kind        VARCHAR(100)  '$.ArtifactKind',
    app_name             VARCHAR(400)  '$.AppName',
    capacity_id          VARCHAR(50)   '$.CapacityId',
    capacity_name        VARCHAR(400)  '$.CapacityName',
    distribution_method  VARCHAR(100)  '$.DistributionMethod',
    consumption_method   VARCHAR(100)  '$.ConsumptionMethod',
    models_snapshots     NVARCHAR(MAX) '$.ModelsSnapshots' AS JSON
) AS evt;
```

Note the last column. `AS JSON` is required for any field that holds an array or
object, and it hands you the raw JSON text. To turn that into rows, run a second
`OPENJSON` over it:

```sql
SELECT
    evt.event_id,
    snapshot.value AS model_snapshot
FROM dbo.pbi_activity AS evt
OUTER APPLY OPENJSON(evt.models_snapshots) AS snapshot;
```

`OUTER APPLY` rather than `CROSS APPLY`, for the same reason as `OUTER => TRUE`
in Snowflake: it keeps events with an empty array.

**Route 2: Lakehouse notebook with Spark.** If the file is in OneLake and you're
working in a Fabric notebook, this is the shorter path, and the SQL is identical
to the Databricks section above because both run Spark:

```python
df = (spark.read
        .option("multiLine", "true")
        .json("Files/activity_events.json"))

df.write.mode("overwrite").saveAsTable("pbi_activity")
```

---

## 3. Keep only report-related activity

Before filtering, one correction that will save you time.

**There is no page-view operation in the activity log.** Power BI records that a
report was opened, via `ViewReport`, but it does not emit an event each time a
user moves to a different page within that report. Microsoft states this plainly
in the usage metrics documentation: "Certain metrics in usage metrics report
aren't included in audit logs. For example, report page views aren't part of
audit logs."

This surprises people, because the built-in **usage metrics report does** show a
per-page breakdown. Both things are true, because they come from two completely
different pipelines. Section 3.1 below explains the difference and what to do
about it.

**What you can tell apart here** is where the report was viewed from. The
`DistributionMethod` field says `Workspace` or `Apps`, which is exactly the
app-versus-workspace split. `ConsumptionMethod` adds the surface, such as
`Power BI Web` or `Power BI Mobile`. Those two fields answer most of what people
actually want when they ask about app views.

In [ ]:
REPORT_OPERATIONS = {
    # Consumption
    "ViewReport", "ViewTile", "ViewDashboard", "ViewUsageMetrics",
    "ReadArtifact", "GetSnapshots", "AnalyzeInExcelReport",
    # Export and print
    "ExportReport", "PrintReport", "DownloadReport", "ExportTile",
    "ExportArtifact", "ExportArtifactDownload", "ExportScorecard",
    # Authoring
    "CreateReport", "CreateReportFromLakehouse", "EditReport",
    "EditReportDescription", "EditReportProperties", "UpdateReportContent",
    "RenameReport", "DeleteReport", "CopyReport", "RebindReport",
    "SaveAutogeneratedReport", "UpgradeReportToPBIR",
    # Sharing and distribution
    "ShareReport", "PublishToWebReport", "PinReportToTeamsChannel",
    "InstallTeamsAnalyticsReport",
}

reports = events[events["Operation"].isin(REPORT_OPERATIONS)].copy()

print(f"{len(reports)} of {len(events)} events are report-related")
print()
print(reports["Operation"].value_counts().to_string())

That filter is deliberately based on the operation name. A second, looser filter
is sometimes more useful: catch anything that *touched* a report, regardless of
what the operation was called, by testing whether the record carries a report
identifier at all. `ReadArtifact` is the reason this matters, because it fires for
many artifact kinds and only the `ArtifactKind` field tells you it was a report.

In [ ]:
touched_report = (
    events["ReportId"].notna()
    | events.get("ArtifactKind", pd.Series(index=events.index, dtype=object)).eq("Report")
)

print(f"{touched_report.sum()} events carry a report identifier or a Report artifact kind")
print()
print(
    events.loc[touched_report, ["Operation", "ArtifactKind"]]
    .value_counts(dropna=False)
    .to_string()
)

### Report views split by app versus workspace

This is the breakdown that the `DistributionMethod` field makes possible.

In [ ]:
views = events[events["Operation"] == "ViewReport"]

if len(views):
    summary = (
        views.groupby(["DistributionMethod", "ConsumptionMethod"], dropna=False)
        .size()
        .reset_index(name="views")
        .sort_values("views", ascending=False)
    )
    print(summary.to_string(index=False))
    print()
    print("Most-viewed reports:")
    print(
        views.groupby(["ReportName", "WorkSpaceName"], dropna=False)
        .size()
        .reset_index(name="views")
        .sort_values("views", ascending=False)
        .head(10)
        .to_string(index=False)
    )
else:
    print("no ViewReport events in this batch")

### The same filter in SQL

Identical logic on all three platforms, since it's just a `WHERE` clause. The only
difference is the string-quoting style, and none of these need anything exotic.

```sql
SELECT
    creation_time_utc,
    user_id,
    operation,
    report_name,
    workspace_name,
    distribution_method,
    consumption_method
FROM pbi_activity
WHERE operation IN (
        'ViewReport', 'ViewTile', 'ViewDashboard', 'ViewUsageMetrics',
        'ReadArtifact', 'GetSnapshots', 'AnalyzeInExcelReport',
        'ExportReport', 'PrintReport', 'DownloadReport', 'ExportTile',
        'ExportArtifact', 'ExportArtifactDownload', 'ExportScorecard',
        'CreateReport', 'CreateReportFromLakehouse', 'EditReport',
        'EditReportDescription', 'EditReportProperties', 'UpdateReportContent',
        'RenameReport', 'DeleteReport', 'CopyReport', 'RebindReport',
        'SaveAutogeneratedReport', 'UpgradeReportToPBIR',
        'ShareReport', 'PublishToWebReport', 'PinReportToTeamsChannel',
        'InstallTeamsAnalyticsReport'
      )
   OR report_id IS NOT NULL
   OR artifact_kind = 'Report'
ORDER BY creation_time_utc DESC;
```

And the app-versus-workspace split:

```sql
SELECT
    distribution_method,
    consumption_method,
    COUNT(*)                    AS views,
    COUNT(DISTINCT user_id)     AS distinct_users
FROM pbi_activity
WHERE operation = 'ViewReport'
GROUP BY distribution_method, consumption_method
ORDER BY views DESC;
```

### 3.1 Why page views are missing here, and where they actually come from

The usage metrics report can break views down by page. The activity log cannot.
That difference is not a gap in this script, it is two separate telemetry systems
with different collection points.

#### Why the server never sees a page change

When someone opens a report, the browser asks the service for it, and the service
records a `ViewReport` event. When that person then clicks through to page two,
nothing is requested from the server. The page definitions were already delivered
to the browser when the report first loaded. Microsoft's FAQ puts it this way:

> Report Views rely on server telemetry that is generated when the report is
> first opened. Once a report is open, its page definitions are already loaded
> onto the user's device.

So the server has nothing to log. Page-level data exists only because the browser
itself reports back, separately, into an internal usage store. From the
report-level auditing guidance:

> The report page views are obtained by using client telemetry, which has
> limitations. Client telemetry (used by report usage metrics) is different from
> server-side telemetry data (used by the activity log).

#### The two pipelines side by side

| | Activity log (this notebook) | Usage metrics page views |
|---|---|---|
| Collected by | The Power BI service, server-side | The user's browser or device |
| Reaches you through | Admin activity events REST API | Usage Metrics Report semantic model only |
| Page-level detail | No | Yes |
| Scope | Whole tenant | One workspace at a time |
| History | 28 days via the API, keep what you extract | 30 days, then deleted |
| Completeness | Authoritative | Lossy, see below |

#### Page view counts are not reliable, and Microsoft says so

This is the part worth knowing before anyone builds a target around page views.
Because the numbers depend on the client successfully sending data, several
ordinary things silently drop events:

> Report Page Views rely on client telemetry and can be affected by undercounting
> and overcounting of activities due to inconsistent network connections, ad
> blockers, or other client-side issues. Report View metrics relies on activity
> data collected from Power BI service, and matches the aggregate counts of
> activities in audit logs and activity logs.

That last sentence is the reassuring half. Report-level counts reconcile between
the two systems, so the `ViewReport` totals from this notebook should agree with
the usage metrics report. Only the page layer diverges.

Three documented limitations bear directly on the app-versus-workspace analysis
above:

- "Pages for App reports can't be seen in the Report pages table." So the page
  breakdown is missing for exactly the audience that consumes through an app.
- "There are fields in the *Report page views* and *Report load times* tables that
  are always blank."
- Page views can come in *lower* than report views, which looks like a bug and
  isn't. There is a dedicated FAQ entry about it.

#### One thing that inflates your ViewReport counts

Worth knowing when a report looks more popular than it is:

> The report views count is influenced by subscriptions running on the reports.
> When the subscription service captures a snapshot of the report for emails, it
> triggers a flow that logs a ViewReport event.

So scheduled email subscriptions generate `ViewReport` events with no human
involved. If a report has subscriptions, filter or annotate accordingly.

#### If you want page data in your own pipeline

There is no admin API for it. Page views live only in the per-workspace **Usage
Metrics Report** semantic model, which "imports data from a Power BI internal
usage metrics store by using a custom Usage Metrics Data Connector."

To get at it:

1. In the workspace, open a report, then **More options** and **Open usage
   metrics**. This creates the semantic model on first use.
2. Connect to that semantic model from Power BI Desktop, via **Get Data**,
   **Power BI semantic models**, or use **Analyze in Excel**.
3. Export on a schedule. This part is not optional if you care about history:
   "The Usage Metrics Report semantic model contains usage data for the last 30
   days only. Data older than 30 days is automatically removed."

Practical constraints to plan around:

- You need Contributor, Member or Admin on the workspace. Viewer is not enough.
- It is one workspace at a time. There is no tenant-wide version.
- Refresh is daily and you cannot trigger it manually. New activity can take up
  to 24 hours to appear.
- It is not supported in My Workspace.
- The semantic model is in preview and Microsoft warns that changes to it may
  break custom reports built on top.

Microsoft's own guidance is to use the activity log, the thing this notebook
reads, whenever you want more than one workspace, more than 30 days of history,
or the full range of user activity. Reach for usage metrics when you specifically
need page-level detail or report opening times, and treat those numbers as
indicative rather than exact.

#### Tracking who is using usage metrics

A small bonus. The activity log does record when someone opens a usage metrics
report, through the `ViewUsageMetrics` operation, which is already in the
"Report - consumption" bucket in section 4.

```python
events[events["Operation"] == "ViewUsageMetrics"][["CreationTime", "UserId"]]
```

#### Sources for this section

- [Monitor usage metrics in Power BI workspaces](https://learn.microsoft.com/en-us/power-bi/collaborate-share/service-modern-usage-metrics)
  covers the metric definitions, the considerations and limitations, and the FAQ
  quoted above.
- [Power BI implementation planning: report-level auditing](https://learn.microsoft.com/en-us/power-bi/guidance/powerbi-implementation-planning-auditing-monitoring-report-level-auditing)
  is the source for the client-versus-server telemetry distinction and the
  guidance on when to prefer the activity log.
- [Track user activities in Power BI](https://learn.microsoft.com/en-us/fabric/enterprise/powerbi/service-admin-auditing)
  describes the activity log itself.
- [Monitor report usage metrics](https://learn.microsoft.com/en-us/power-bi/collaborate-share/service-usage-metrics)
  documents the older usage metrics report that the improved one replaces.
- [Audit and usage admin settings](https://learn.microsoft.com/en-us/fabric/admin/service-admin-portal-audit-usage)
  covers the two tenant settings that control usage metrics and whether user
  names appear in them.

---

## 4. Group activities into buckets

**On the official documentation.** Microsoft publishes the complete list of
operation names with a friendly description for each, at
[learn.microsoft.com/fabric/admin/operation-list](https://learn.microsoft.com/en-us/fabric/admin/operation-list).
That page is the authoritative source for *which operations exist* and *what each
one means*, and there are around 740 of them.

What it does **not** publish is a grouping. The page is one flat alphabetical
table with no categories, so there is no official set of buckets to adopt. The
buckets below are therefore mine, but every operation name in them has been checked
against that official list, and the check is repeated in code further down so you
can see for yourself.

In [ ]:
ACTIVITY_BUCKETS = {
    "Report - consumption": [
        "ViewReport", "ViewTile", "ViewDashboard", "ViewUsageMetrics",
        "ReadArtifact", "GetSnapshots", "AnalyzeInExcelReport",
    ],
    "Report - export & print": [
        "ExportReport", "PrintReport", "DownloadReport", "ExportTile",
        "ExportArtifact", "ExportArtifactDownload", "ExportScorecard",
    ],
    "Report - authoring": [
        "CreateReport", "CreateReportFromLakehouse", "EditReport",
        "EditReportDescription", "EditReportProperties", "UpdateReportContent",
        "RenameReport", "DeleteReport", "CopyReport", "RebindReport",
        "SaveAutogeneratedReport", "UpgradeReportToPBIR",
    ],
    "Report - sharing & distribution": [
        "ShareReport", "PublishToWebReport", "PinReportToTeamsChannel",
        "PinReportGetChannelsInTeam", "PinReportGetUserJoinedTeams",
        "InstallTeamsAnalyticsReport",
    ],
    "Dashboard": [
        "CreateDashboard", "EditDashboard", "DeleteDashboard", "CopyDashboard",
        "RenameDashboard", "ShareDashboard",
        "AddTile", "EditTile", "DeleteTile", "CloneTile", "EditWidgetTile",
    ],
    "Semantic model (dataset)": [
        "CreateDataset", "DeleteDataset", "EditDataset", "RefreshDataset",
        "SetScheduledRefresh", "BindToGateway", "UpdateDatasetParameters",
        "TakeOverDataset", "AnalyzedByExternalApplication",
    ],
    "App & template app": [
        "CreateApp", "UpdateApp", "InstallApp", "CreateOrgApp", "DeleteOrgApp",
        "CreateTemplateApp", "CreateTemplateAppPackage", "DeleteTemplateApp",
        "DeleteTemplateAppPackage", "InstallTemplateApp",
        "UpdateTemplateAppSettings", "UpdateTemplateAppTestPackagePermissions",
        "ExtractTemplateAppPackage", "CreateTemplateAppInstallTicket",
    ],
    "Workspace & capacity": [
        "CreateFolder", "DeleteFolder", "UpdateFolder", "AddFolderAccess",
        "DeleteFolderAccess", "UpdateFolderAccess", "MigrateWorkspaceIntoCapacity",
        "ModifyWorkspaceCapacity", "RemoveWorkspacesFromCapacity",
        "ChangeCapacityState", "UpdateCapacityUsersAssignment",
    ],
    "Admin & governance": [
        "UpdatedAdminFeatureSwitch", "ExportActivityEvents",
        "AddAdminPersonalWorkspaceAccess", "GetReportsAsAdmin",
        "GetDatasetsAsAdmin", "GetDashboardsAsAdmin", "GetAppsAsAdmin",
    ],
}

OPERATION_TO_BUCKET = {
    op: bucket for bucket, ops in ACTIVITY_BUCKETS.items() for op in ops
}

print(f"{len(ACTIVITY_BUCKETS)} buckets covering {len(OPERATION_TO_BUCKET)} operations")

### Verify every name against Microsoft's list

This is the part that keeps the mapping honest. Any operation name that isn't in
the official reference gets printed, so a typo or a name I invented shows up
immediately rather than silently producing an empty bucket.

In [ ]:
if REFERENCE.exists():
    reference = pd.read_csv(REFERENCE)
    official = set(reference["operation_name"])
    print(f"official catalogue: {len(official)} operations\n")

    unknown = sorted(set(OPERATION_TO_BUCKET) - official)
    if unknown:
        print("In my buckets but NOT in Microsoft's published list:")
        for op in unknown:
            print(f"  - {op}")
        print("\n(These are real operations seen in live data. Microsoft's page")
        print(" lags the service, so newer operations appear before they're documented.)")
    else:
        print("Every bucketed operation appears in the official list.")
else:
    reference = None
    print(f"{REFERENCE} not found; skipping verification")

### Apply the buckets

Anything that doesn't match a bucket is labelled rather than dropped. Silently
discarding unmatched rows is how you end up confidently reporting on half your data.

In [ ]:
events["ActivityBucket"] = events["Operation"].map(OPERATION_TO_BUCKET).fillna("Other / unclassified")

bucket_summary = (
    events.groupby("ActivityBucket")
    .agg(events=("Id", "size"),
         users=("UserId", "nunique"),
         first_seen=("CreationTime", "min"),
         last_seen=("CreationTime", "max"))
    .sort_values("events", ascending=False)
)
print(bucket_summary.to_string())

In [ ]:
unclassified = events.loc[events["ActivityBucket"] == "Other / unclassified", "Operation"]

if len(unclassified):
    print("Operations landing in 'Other / unclassified':\n")
    for op, n in unclassified.value_counts().items():
        friendly = ""
        if reference is not None:
            match = reference.loc[reference["operation_name"] == op, "friendly_name"]
            friendly = f"  ({match.iloc[0]})" if len(match) else "  (not in Microsoft's list)"
        print(f"  {n:>4}  {op}{friendly}")
else:
    print("Everything in this batch was classified.")

### Attach Microsoft's friendly descriptions

Joining the official reference onto the events gives every row a plain-English
label straight from the documentation, which makes the output readable by people
who don't know the operation names.

In [ ]:
if reference is not None:
    labelled = events.merge(
        reference.rename(columns={"operation_name": "Operation",
                                  "friendly_name": "OperationDescription"}),
        on="Operation", how="left",
    )
    print(
        labelled[["Operation", "OperationDescription", "ActivityBucket"]]
        .drop_duplicates()
        .sort_values(["ActivityBucket", "Operation"])
        .to_string(index=False)
    )

### The same bucketing in SQL

A `CASE` expression is the portable way to do this, and it runs unchanged on all
three platforms. For a mapping this size, though, a lookup table is easier to
maintain than a wall of `WHEN` clauses. Load `powerbi_operations_reference.csv`
plus your bucket assignments as a dimension table and join to it, and you can
change the grouping without editing any queries.

```sql
SELECT
    CASE
        WHEN operation IN ('ViewReport', 'ViewTile', 'ViewDashboard',
                           'ViewUsageMetrics', 'ReadArtifact', 'GetSnapshots',
                           'AnalyzeInExcelReport')
            THEN 'Report - consumption'
        WHEN operation IN ('ExportReport', 'PrintReport', 'DownloadReport',
                           'ExportTile', 'ExportArtifact', 'ExportArtifactDownload',
                           'ExportScorecard')
            THEN 'Report - export & print'
        WHEN operation IN ('CreateReport', 'CreateReportFromLakehouse', 'EditReport',
                           'EditReportDescription', 'EditReportProperties',
                           'UpdateReportContent', 'RenameReport', 'DeleteReport',
                           'CopyReport', 'RebindReport', 'SaveAutogeneratedReport',
                           'UpgradeReportToPBIR')
            THEN 'Report - authoring'
        WHEN operation IN ('ShareReport', 'PublishToWebReport',
                           'PinReportToTeamsChannel', 'InstallTeamsAnalyticsReport')
            THEN 'Report - sharing & distribution'
        WHEN operation LIKE '%Dashboard%' OR operation LIKE '%Tile%'
            THEN 'Dashboard'
        WHEN operation LIKE '%Dataset%' OR operation LIKE '%Refresh%'
            THEN 'Semantic model (dataset)'
        WHEN operation LIKE '%TemplateApp%' OR operation LIKE '%App%'
            THEN 'App & template app'
        WHEN operation LIKE '%Capacity%' OR operation LIKE '%Folder%'
            THEN 'Workspace & capacity'
        WHEN operation LIKE '%AsAdmin' OR operation LIKE 'Update%AdminFeature%'
            THEN 'Admin & governance'
        ELSE 'Other / unclassified'
    END                             AS activity_bucket,
    COUNT(*)                        AS events,
    COUNT(DISTINCT user_id)         AS distinct_users
FROM pbi_activity
GROUP BY 1
ORDER BY events DESC;
```

The `LIKE` fallbacks at the bottom are a safety net for operations that appear
after you write this. They're a blunt instrument and will occasionally misfile
something, so check the `Other / unclassified` bucket periodically and promote
anything that shows up repeatedly into an explicit list.

---

## 5. Save the results

In [ ]:
events.to_csv("activity_events_bucketed.csv", index=False)
print("wrote activity_events_bucketed.csv")

reports_out = events[events["ActivityBucket"].str.startswith("Report")]
reports_out.to_csv("activity_events_reports_only.csv", index=False)
print(f"wrote activity_events_reports_only.csv ({len(reports_out)} rows)")

## Things worth remembering

- **All timestamps are UTC**, with no timezone marker on them. Convert once, early.
- **There are no page-view events in the activity log.** `ViewReport` fires when a
  report opens, not per page. Page-level data exists only in the usage metrics
  semantic model, from a separate client-side pipeline. See section 3.1.
- **Use `DistributionMethod`** to separate app views from workspace views. This is
  the one split the activity log does give you, and the usage metrics report
  cannot show pages for app reports at all.
- **Subscriptions inflate `ViewReport`.** Each emailed snapshot logs a view with
  no human involved.
- **Fields vary by operation.** Any flat table of this data is mostly nulls, which
  is normal and not a sign of a broken extract.
- **The operation list grows.** New names appear in live data before they reach the
  documentation, so keep an `Other / unclassified` bucket and check it now and then.
- **`ReadArtifact` is not report-specific.** It fires for many item types, so read
  `ArtifactKind` alongside it before counting it as report usage.

---

## Microsoft documentation used in this notebook

**Activity log, the source of `activity_events.json`**

- [Operation list](https://learn.microsoft.com/en-us/fabric/admin/operation-list)
  is the authoritative catalogue of operation names and their friendly
  descriptions, extracted to `powerbi_operations_reference.csv`.
- [Admin - Get Activity Events REST API](https://learn.microsoft.com/en-us/rest/api/power-bi/admin/get-activity-events)
  is the endpoint `smoke_test.py` calls, including the same-UTC-day and
  continuation token rules.
- [Track user activities in Power BI](https://learn.microsoft.com/en-us/fabric/enterprise/powerbi/service-admin-auditing)
  explains the activity log and how it relates to the Microsoft Purview audit log.
- [Access the Power BI activity log](https://learn.microsoft.com/en-us/power-bi/guidance/admin-activity-log)
  is the guidance article on extracting and retaining activity data.
- [Power BI implementation planning: tenant-level auditing](https://learn.microsoft.com/en-us/power-bi/guidance/powerbi-implementation-planning-auditing-monitoring-tenant-level-auditing)
  covers building a durable activity log extract.

**Usage metrics, the source of page-level views**

- [Monitor usage metrics in Power BI workspaces](https://learn.microsoft.com/en-us/power-bi/collaborate-share/service-modern-usage-metrics)
  defines Report Views against Report Page Views, and lists the limitations that
  make page counts approximate.
- [Power BI implementation planning: report-level auditing](https://learn.microsoft.com/en-us/power-bi/guidance/powerbi-implementation-planning-auditing-monitoring-report-level-auditing)
  distinguishes client telemetry from server-side telemetry and says when to
  prefer the activity log.
- [Monitor report usage metrics](https://learn.microsoft.com/en-us/power-bi/collaborate-share/service-usage-metrics)
  documents the older usage metrics report.
- [Audit and usage admin settings](https://learn.microsoft.com/en-us/fabric/admin/service-admin-portal-audit-usage)
  covers the tenant settings for usage metrics and per-user data.